In [ ]:
#必要なモジュールのインポート
import boto3 # AWSのサービスを利用するときに必須
import json # json形式のデータを処理する
from PIL import Image # pillow の Image モジュール 画像処理に使用する
from PIL import ImageDraw

In [ ]:
file_in_1 = 'a-chan_2024.jpg' # この画像の顔を使う
file_in_2 = 'mainvisual_pc202406.jpg' # この画像から探す
file_out = 'compared.jpg' # 見つかった顔を赤枠で囲んで出力する画像ファイル名

In [ ]:
# Rekognition サービスクライアントを作成
rekognition = boto3.client('rekognition')

# ソース画像(元になる顔)をオープン
with open(file_in_1, 'rb') as source:
    # ターゲット画像(この画像から該当の顔を探す)をオープン
    with open(file_in_2, 'rb') as target:
        # ソース画像の顔をターゲット画像から探す
        result = rekognition.compare_faces(
            SourceImage={'Bytes': source.read()}, # 元になる顔
            TargetImage={'Bytes': target.read()}) # ここから顔を探す

# 検出結果のjsonを整形して表示
print(json.dumps(result, indent=4))

In [ ]:
# ターゲット画像(この画像から該当の顔を探す)をオープン
image_in = Image.open(file_in_2)
# 画像のサイズを取得
w, h = image_in.size

draw = ImageDraw.Draw(image_in)

# 検出された顔の分だけ繰り返す
for face in result['FaceMatches']: # 見つかったものはFaceMatchesに含まれる
    # その中のキーFaceに情報が入っている
    # バウンディングボックスを取得
    box = face['Face']['BoundingBox']

    # 顔の左、上、右、下の座標を取得
    left = int(box['Left']*w)
    top = int(box['Top']*h)
    right = left+int(box['Width']*w)
    bottom = top+int(box['Height']*h)

    # 入力画像から出力画像に顔の部分を貼り付け
    draw.rectangle( (left, top, right, bottom), fill=None, outline=(255,0,0), width=10)

# 画像をファイルに保存
image_in.save(file_out)

# 画像を表示
image_in.show()